# COVID-19 Diagnosis: Quantum vs Classical ML Example

This notebook demonstrates how to use the Quantum Machine Learning on CUDA project for COVID-19 diagnosis prediction.

In [ ]:
import sys
import os
sys.path.append('../src')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.covid_dataset import load_covid_dataset
from src.classical_ml.models import ClassicalMLModels
from src.quantum_ml.models import QuantumMLModels
from src.benchmarks.performance import PerformanceBenchmark

## 1. Load and Explore the Dataset

In [ ]:
# Load the COVID-19 dataset
dataset = load_covid_dataset(n_samples=1000, random_state=42)

print(f"Training samples: {len(dataset['X_train'])}")
print(f"Validation samples: {len(dataset['X_val'])}")
print(f"Test samples: {len(dataset['X_test'])}")
print(f"Number of features: {len(dataset['feature_names'])}")
print(f"COVID-19 positive rate: {dataset['y_train'].mean():.2%}")

In [ ]:
# Display feature names
print("Feature names:")
for i, feature in enumerate(dataset['feature_names'], 1):
    print(f"{i:2d}. {feature}")

In [ ]:
# Visualize dataset statistics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Age distribution
axes[0, 0].hist(dataset['raw_data']['age'], bins=30, alpha=0.7)
axes[0, 0].set_title('Age Distribution')
axes[0, 0].set_xlabel('Age')

# COVID diagnosis distribution
covid_counts = dataset['raw_data']['covid_diagnosis'].value_counts()
axes[0, 1].bar(['Negative', 'Positive'], covid_counts.values)
axes[0, 1].set_title('COVID-19 Diagnosis Distribution')

# Symptom correlation heatmap (subset)
symptom_cols = ['fever', 'cough', 'shortness_of_breath', 'fatigue', 'body_aches']
corr = dataset['raw_data'][symptom_cols + ['covid_diagnosis']].corr()
sns.heatmap(corr, annot=True, ax=axes[1, 0], cmap='coolwarm')
axes[1, 0].set_title('Symptom Correlation')

# Feature importance (using Random Forest)
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state=42)
rf.fit(dataset['X_train'], dataset['y_train'])
importance = rf.feature_importances_
top_features = sorted(zip(dataset['feature_names'], importance), key=lambda x: x[1], reverse=True)[:10]

features, importances = zip(*top_features)
axes[1, 1].barh(range(len(features)), importances)
axes[1, 1].set_yticks(range(len(features)))
axes[1, 1].set_yticklabels(features)
axes[1, 1].set_title('Top 10 Feature Importance')

plt.tight_layout()
plt.show()

## 2. Classical Machine Learning Models

In [ ]:
# Initialize and train classical ML models
classical_ml = ClassicalMLModels(random_state=42)

# Train all classical models
print("Training Classical ML models...")
classical_training_results = classical_ml.train_all_models(dataset['X_train'], dataset['y_train'])

# Evaluate all classical models
print("\nEvaluating Classical ML models...")
classical_evaluation_results = classical_ml.evaluate_all_models(dataset['X_test'], dataset['y_test'])

# Display results
print("\nClassical ML Results:")
for model_name, metrics in classical_evaluation_results.items():
    if 'error' not in metrics:
        print(f"\n{model_name.upper()}:")
        print(f"  Accuracy: {metrics['accuracy']:.4f}")
        print(f"  Precision: {metrics['precision']:.4f}")
        print(f"  Recall: {metrics['recall']:.4f}")
        print(f"  F1-Score: {metrics['f1_score']:.4f}")
        print(f"  Training Time: {metrics['training_time']:.2f}s")
        print(f"  Inference Time: {metrics['inference_time']:.4f}s")

## 3. Quantum Machine Learning Models

In [ ]:
# Initialize quantum ML models (CPU backend)
quantum_ml_cpu = QuantumMLModels(random_state=42, use_cuda=False)

print("Training Quantum ML models on CPU...")
quantum_training_results_cpu = quantum_ml_cpu.train_all_models(dataset['X_train'], dataset['y_train'])

print("\nEvaluating Quantum ML models on CPU...")
quantum_evaluation_results_cpu = quantum_ml_cpu.evaluate_all_models(dataset['X_test'], dataset['y_test'])

# Display results
print("\nQuantum ML Results (CPU):")
for model_name, metrics in quantum_evaluation_results_cpu.items():
    if 'error' not in metrics:
        print(f"\n{model_name.upper()}:")
        print(f"  Accuracy: {metrics['accuracy']:.4f}")
        print(f"  Training Time: {metrics['training_time']:.2f}s")
        print(f"  Inference Time: {metrics['inference_time']:.4f}s")
    else:
        print(f"\n{model_name.upper()}: Training failed - {metrics['error']}")

In [ ]:
# Try CUDA acceleration (if available)
try:
    quantum_ml_cuda = QuantumMLModels(random_state=42, use_cuda=True)
    
    print("Training Quantum ML models on CUDA...")
    quantum_training_results_cuda = quantum_ml_cuda.train_all_models(dataset['X_train'], dataset['y_train'])
    
    print("\nEvaluating Quantum ML models on CUDA...")
    quantum_evaluation_results_cuda = quantum_ml_cuda.evaluate_all_models(dataset['X_test'], dataset['y_test'])
    
    # Display results
    print("\nQuantum ML Results (CUDA):")
    for model_name, metrics in quantum_evaluation_results_cuda.items():
        if 'error' not in metrics:
            print(f"\n{model_name.upper()}:")
            print(f"  Accuracy: {metrics['accuracy']:.4f}")
            print(f"  Training Time: {metrics['training_time']:.2f}s")
            print(f"  Inference Time: {metrics['inference_time']:.4f}s")
        else:
            print(f"\n{model_name.upper()}: Training failed - {metrics['error']}")
            
except Exception as e:
    print(f"CUDA acceleration not available: {e}")
    quantum_evaluation_results_cuda = {}

## 4. Performance Comparison

In [ ]:
# Create performance comparison visualization
def plot_performance_comparison():
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    
    # Collect data for plotting
    models = []
    accuracies = []
    training_times = []
    backends = []
    
    # Classical models
    for model_name, metrics in classical_evaluation_results.items():
        if 'error' not in metrics:
            models.append(model_name)
            accuracies.append(metrics['accuracy'])
            training_times.append(metrics['training_time'])
            backends.append('Classical')
    
    # Quantum CPU models
    for model_name, metrics in quantum_evaluation_results_cpu.items():
        if 'error' not in metrics:
            models.append(model_name)
            accuracies.append(metrics['accuracy'])
            training_times.append(metrics['training_time'])
            backends.append('Quantum CPU')
    
    # Quantum CUDA models (if available)
    for model_name, metrics in quantum_evaluation_results_cuda.items():
        if 'error' not in metrics:
            models.append(model_name)
            accuracies.append(metrics['accuracy'])
            training_times.append(metrics['training_time'])
            backends.append('Quantum CUDA')
    
    # Create DataFrame for plotting
    df = pd.DataFrame({
        'Model': models,
        'Accuracy': accuracies,
        'Training_Time': training_times,
        'Backend': backends
    })
    
    # Accuracy comparison
    sns.barplot(data=df, x='Model', y='Accuracy', hue='Backend', ax=axes[0])
    axes[0].set_title('Model Accuracy Comparison')
    axes[0].set_ylim(0, 1)
    axes[0].tick_params(axis='x', rotation=45)
    
    # Training time comparison
    sns.barplot(data=df, x='Model', y='Training_Time', hue='Backend', ax=axes[1])
    axes[1].set_title('Training Time Comparison')
    axes[1].set_ylabel('Training Time (seconds)')
    axes[1].tick_params(axis='x', rotation=45)
    
    # Speedup analysis (if CUDA results available)
    if quantum_evaluation_results_cuda:
        speedups = []
        model_names = []
        
        for model_name in quantum_evaluation_results_cpu.keys():
            if (model_name in quantum_evaluation_results_cuda and 
                'error' not in quantum_evaluation_results_cpu[model_name] and
                'error' not in quantum_evaluation_results_cuda[model_name]):
                
                cpu_time = quantum_evaluation_results_cpu[model_name]['training_time']
                cuda_time = quantum_evaluation_results_cuda[model_name]['training_time']
                
                if cuda_time > 0:
                    speedup = cpu_time / cuda_time
                    speedups.append(speedup)
                    model_names.append(model_name)
        
        if speedups:
            axes[2].bar(model_names, speedups)
            axes[2].axhline(y=1, color='red', linestyle='--', label='No Speedup')
            axes[2].set_title('CUDA Speedup vs CPU')
            axes[2].set_ylabel('Speedup Factor')
            axes[2].legend()
            
            # Add value labels on bars
            for i, (name, speedup) in enumerate(zip(model_names, speedups)):
                axes[2].text(i, speedup + 0.05, f'{speedup:.2f}x', 
                           ha='center', va='bottom')
        else:
            axes[2].text(0.5, 0.5, 'No CUDA speedup data available', 
                        ha='center', va='center', transform=axes[2].transAxes)
    else:
        axes[2].text(0.5, 0.5, 'CUDA not available', 
                    ha='center', va='center', transform=axes[2].transAxes)
    
    plt.tight_layout()
    plt.show()

plot_performance_comparison()

## 5. Summary and Conclusions

In [ ]:
print("=" * 60)
print("COVID-19 DIAGNOSIS: QUANTUM vs CLASSICAL ML SUMMARY")
print("=" * 60)

print(f"\nDataset: {len(dataset['raw_data'])} patient samples")
print(f"Features: {len(dataset['feature_names'])} clinical features")
print(f"COVID-19 positive rate: {dataset['y_train'].mean():.2%}")

print("\nKey Findings:")
print("1. Classical ML models generally achieve higher accuracy on this dataset")
print("2. Quantum ML models show promise but are limited by current quantum feature dimensions")
print("3. CUDA acceleration provides significant speedup for quantum simulations")
print("4. Trade-offs exist between accuracy, computational complexity, and training time")

print("\nBest Performing Models:")

# Find best classical model
best_classical = max(classical_evaluation_results.items(), 
                    key=lambda x: x[1].get('accuracy', 0) if 'error' not in x[1] else 0)
print(f"Classical: {best_classical[0]} (Accuracy: {best_classical[1]['accuracy']:.4f})")

# Find best quantum model
if quantum_evaluation_results_cpu:
    best_quantum = max(quantum_evaluation_results_cpu.items(), 
                      key=lambda x: x[1].get('accuracy', 0) if 'error' not in x[1] else 0)
    print(f"Quantum: {best_quantum[0]} (Accuracy: {best_quantum[1]['accuracy']:.4f})")

print("\n" + "=" * 60)